In [5]:
from net2brain.feature_extraction import FeatureExtractor
import torch
import open_clip

import cv2
import os, glob
import numpy as np
import joblib
from pathlib import Path

In [ ]:
def video_for(stim_id):
    target_name = f'{int(stim_id):04d}.mp4'
    match = next((v for v in vids if v.name == target_name), None)
    assert match is not None, f'Video for stimulus {stim_id} not found!'
    return str(match)

def dump_frames_for_batch(stim_ids, out_dir: Path):
    for f in out_dir.glob('*.jpg'):
        f.unlink()
    out_dir.mkdir(parents=True, exist_ok=True)
    
    for sid in stim_ids:
        cap = cv2.VideoCapture(video_for(sid))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        picks = np.linspace(0, max(total - 1, 0), N_FRAMES).astype(int)
        for k, fno in enumerate(picks):
            # Use Path division operator for frame saving
            path = out_dir / f'{int(sid):05d}_f{k:02d}.jpg'
            if path.exists():
                continue
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(fno))
            ok, frame = cap.read()
            if ok:
                cv2.imwrite(str(path), frame)
        cap.release()

def consolidate_to_video_level(raw_npz_path, output_path):
        z = np.load(raw_npz_path)
        video_features = {}
        
        all_sids = set(k.split('_')[0] for k in z.files)
        
        for sid in all_sids:
            keys = [k for k in z.files if k.startswith(f'{sid}_')]
            frame_feats = [z[k].flatten() for k in keys]
            video_features[sid] = np.mean(frame_feats, axis=0)
            
        # Save as a clean video-level archive
        np.savez_compressed(output_path, **video_features)
        print(f"Saved video-level features for {len(video_features)} videos to {output_path}")

# Process videos and extract Alex, Dino, and OpenCLIP features
## Store them under eeg-mind-reading-challenge/artifacts

## 1. PREPARE TRAINING SET

In [17]:

BASE_DIR     = Path('../eeg-mind-reading-challenge')
DATA_DIR     = Path('../data')
STIM_DIR     = BASE_DIR / 'stimuli_train'  
TRAIN_DIR    = BASE_DIR / 'trn'
FEAT_DIR     = DATA_DIR / 'features'       
FRAME_DIR    = DATA_DIR / 'frames'   
TEMP_FEAT    = DATA_DIR / 'temp_features' 

N_FRAMES   = 8                                             
BATCH_SIZE = 200                                         

os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(TEMP_FEAT, exist_ok=True)
os.makedirs(FRAME_DIR, exist_ok=True)

vids = sorted(list(STIM_DIR.glob('*.mp4')))
print(f"Found {len(vids)} total videos in {STIM_DIR}")
all_stim_ids = np.arange(1, len(vids) + 1)

fxs = []
LAYERS = []

fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')   # 'cuda' if available
LAYER = 'features.3'
fxs.append(fx)
LAYERS.append(LAYER)

dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval()
fx = FeatureExtractor(model=dino_model, device='cpu')   # 'cuda' if available
LAYER = 'blocks.6'
fxs.append(fx)
LAYERS.append(LAYER)

clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
fx = FeatureExtractor(model=clip_model, device='cpu')
LAYER = 'visual.transformer.resblocks.7'
fxs.append(fx)
LAYERS.append(LAYER)

for fx, LAYER in zip(fxs, LAYERS):
    master_features = {}

    for i in range(0, len(all_stim_ids), BATCH_SIZE):
        batch_ids = all_stim_ids[i:i + BATCH_SIZE]
        print(f"\nProcessing batch: videos {i+1} to {min(i + BATCH_SIZE, len(all_stim_ids))}...")
        
        dump_frames_for_batch(batch_ids, FRAME_DIR)
        
        fx.extract(data_path=FRAME_DIR, layers_to_extract=[LAYER], save_path=TEMP_FEAT)
        
        batch_npz_path = TEMP_FEAT / f'consolidated_{LAYER}.npz'
        if os.path.exists(batch_npz_path):
            z = np.load(batch_npz_path)
            for k in z.files:
                master_features[k] = z[k]
            os.remove(batch_npz_path)  

    # 4. Save the final consolidated .npz file
    final_npz_path = FEAT_DIR / f'consolidated_{LAYER}.npz'
    np.savez_compressed(final_npz_path, **master_features)
    print(f"\nSuccessfully extracted and saved all features to {final_npz_path} without memory crashes!")

    consolidate_to_video_level(FEAT_DIR / f'consolidated_{LAYER}.npz', FEAT_DIR / f'video_{LAYER}.npz')

Found 1000 total videos in ../eeg-mind-reading-challenge/stimuli_train


Using cache found in /Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main



Processing batch: videos 1 to 200...


Processing files: 100%|██████████| 1600/1600 [00:52<00:00, 30.28it/s, subfiles=1/1]


Consolidating data per layer...


...


Processing batch: videos 801 to 1000...


Processing files: 100%|██████████| 1600/1600 [01:19<00:00, 20.16it/s, subfiles=1/1]


Consolidating data per layer...


100%|██████████| 1600/1600 [00:01<00:00, 1548.76it/s]



Successfully extracted and saved all features to ../eeg-mind-reading-challenge/features/consolidated_visual.transformer.resblocks.7.npz without memory crashes!
Saved video-level features for 1000 videos to ../eeg-mind-reading-challenge/features/video_visual.transformer.resblocks.7.npz


## 1. PREPARE TEST SET

In [19]:
BASE_DIR         = Path('../eeg-mind-reading-challenge')
DATA_DIR         = Path('../data')

TEST_DIR         = BASE_DIR / 'tst'
STIM_TEST_DIR    = BASE_DIR / 'stimuli_test'
TEST_STIM        = BASE_DIR / 'tst_stim'
FRAME_TEST_DIR   = DATA_DIR / 'frames_test'
FEAT_TEST_DIR    = DATA_DIR / 'features_test'
TEMP_FEAT        = DATA_DIR / 'temp_features'

N_FRAMES         = 8                                             
BATCH_SIZE       = 200

os.makedirs(FEAT_TEST_DIR, exist_ok=True)
os.makedirs(TEMP_FEAT, exist_ok=True)
os.makedirs(FRAME_TEST_DIR, exist_ok=True)

vids = sorted(list(STIM_TEST_DIR.glob('*.mp4')))
print(len(vids), 'videos found')
all_stim_ids = np.arange(1001, 1001 + len(vids))

fxs = []
LAYERS = []

fx = FeatureExtractor(model='AlexNet', netset='Standard', device='cpu')   # 'cuda' if available
LAYER = 'features.3'
fxs.append(fx)
LAYERS.append(LAYER)

dino_model = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval()
fx = FeatureExtractor(model=dino_model, device='cpu')   # 'cuda' if available
LAYER = 'blocks.6'
fxs.append(fx)
LAYERS.append(LAYER)

clip_model, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')
clip_model.eval()
fx = FeatureExtractor(model=clip_model, device='cpu')
LAYER = 'visual.transformer.resblocks.7'
fxs.append(fx)
LAYERS.append(LAYER)

for fx, LAYER in zip(fxs, LAYERS):
    master_features = {}

    for i in range(0, len(all_stim_ids), BATCH_SIZE):
        batch_ids = all_stim_ids[i:i + BATCH_SIZE]
        print(f"\nProcessing batch: videos {i+1} to {min(i + BATCH_SIZE, len(all_stim_ids))}...")
        
        dump_frames_for_batch(batch_ids, FRAME_TEST_DIR)

        fx.extract(data_path=FRAME_TEST_DIR, layers_to_extract=[LAYER], save_path=TEMP_FEAT)
        
        batch_npz_path = TEMP_FEAT / f'consolidated_{LAYER}.npz'
        if os.path.exists(batch_npz_path):
            z = np.load(batch_npz_path)
            for k in z.files:
                master_features[k] = z[k]
            os.remove(batch_npz_path) 

    final_npz_path = FEAT_TEST_DIR / f'consolidated_{LAYER}.npz'
    np.savez_compressed(final_npz_path, **master_features)
    print(f"\nSuccessfully extracted and saved all features to {final_npz_path} without memory crashes!")

    consolidate_to_video_level(FEAT_TEST_DIR / f'consolidated_{LAYER}.npz', FEAT_TEST_DIR / f'video_{LAYER}.npz')

102 videos found


Using cache found in /Users/yetkin/.cache/torch/hub/facebookresearch_dinov2_main
/Users/yetkin/Desktop/Hackathon/.venv/lib/python3.11/site-packages/net2brain/feature_extraction.py:104: UserWarning: If you add your own model you can also select our own: 
Preprocessing Function (preprocessor) 
Extraction Function (extraction_function) 
Feature Cleaner (feature_cleaner) 
  warnings.warn("If you add your own model you can also select our own: \nPreprocessing Function (preprocessor) \nExtraction Function (extraction_function) \nFeature Cleaner (feature_cleaner) ")



Processing batch: videos 1 to 102...


Processing files: 100%|██████████| 816/816 [00:16<00:00, 49.87it/s, subfiles=1/1]


Consolidating data per layer...


...


Successfully extracted and saved all features to ../data/features_test/consolidated_blocks.6.npz without memory crashes!
Saved video-level features for 102 videos to ../data/features_test/video_blocks.6.npz

Processing batch: videos 1 to 102...


Processing files: 100%|██████████| 816/816 [00:38<00:00, 21.45it/s, subfiles=1/1]


Consolidating data per layer...


100%|██████████| 816/816 [00:00<00:00, 1135.14it/s]



Successfully extracted and saved all features to ../data/features_test/consolidated_visual.transformer.resblocks.7.npz without memory crashes!
Saved video-level features for 102 videos to ../data/features_test/video_visual.transformer.resblocks.7.npz
